# Feature and Model Selection

In [ ]:
import warnings
import joblib
from pathlib import Path
import shutil

import numpy as np
import pandas as pd


import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load / Reload Selection Utility Functions

In [ ]:
from utils2 import selection as sel

----

## Read Config File

In [ ]:
config_path = Path(r'experiments')

# choose between final and development config file
# config_filename = "bin_sel_dev.yml" # development
config_filename = "bin_sel_final_202608.yml" # final

config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - feature  and model selection (final experiment) after code review using dataset for public request',
  'classification_type': 'binary',
  'stage': 'selection',
  'tag': 'final_202608',
  'verbosity': 0,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/EAMC_DPN_Dataset.xlsx'},
 'feature_selection': {'cross_validation': {'k_splits': 4,
   'n_repeats': 10,
   'sort_by': 'auprc'},
  'vif_threshold': 5},
 'figures': {'summary_table_topk': 5}}

#### Set output directory

In [ ]:
outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.experiment.tag 
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments/binary/selection/final_202608


#### Copy config file to output directory

In [ ]:
source = config_path / config_filename
destination = outputdir / config_filename
shutil.copy(source, destination)

PosixPath('experiments/binary/selection/final_202608/bin_sel_final_202608.yml')

## Data Loading

In [ ]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type);
D.df.tail(3)

Data laoding cleaning report written to ../dataset/cleaning_report.txt.


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
184,1,36,0,1.0,1,6.18,1,0,0,1,0,0,1,0,1,3,11.45,49.2,13.79,40.2,41.8,3.60,11.94,8.78,50.3,14.11,43.1,14.95,41.2,42.2,3.7,9.43,7.19,49.9,69,4,56,8,83.0,7.0,0
185,0,60,1,5.0,1,12.20,1,0,1,0,0,1,1,1,1,8,5.03,37.9,0.00,0.0,36.3,4.85,5.05,3.06,53.1,5.58,37.2,0.00,0.0,32.2,4.2,5.09,3.28,53.5,16,11,21,9,46.0,32.0,1
186,0,65,1,15.0,1,7.59,1,1,0,1,0,1,1,1,1,8,0.00,0.0,0.00,0.0,43.2,5.80,0.56,0.20,0.0,0.00,0.0,0.00,0.0,48.1,5.7,0.27,0.11,0.0,39,16,41,23,43.0,44.0,1


Binary Classification Classes:  ['Negative', 'Possible', 'Probable'] vs 'Confirmed'


In [ ]:
dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
len(data_cols), data_cols

(40,
 Index(['SEX', 'AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C', 'HPN', 'PAOD',
        'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR',
        'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L',
        'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R', 'SSC_R', 'SPSA_R',
        'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R',
        'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM',
        'NS', 'CAS'],
       dtype='object'))

### Data Inspection

In [ ]:
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((187, 40), (187,))

In [ ]:
# get number of postive and negative class
print('Confirmed - positive class', y.sum())
print('Non-confirmed - negative class', y.shape[0]-y.sum())

Confirmed - positive class 130
Non-confirmed - negative class 57


In [ ]:
D.classification_df

,Confirmed,Probable,Possible,Any_DPN,Negative
0,130,27,22,179,8


Rest of notebook is discontinued to give more importance to the scripts than to the notebook.